In [64]:
import os
import re
import pandas as pd

# ---------------- CONFIG ----------------
# Make sure this matches your filenames and working directory
DATA_DIR = "."  # current folder

CBSA_MAP = {
    "Atlanta": "12060",        # Atlanta–Sandy Springs–Alpharetta, GA
    "Chicago": "16980",        # Chicago–Naperville–Elgin, IL–IN–WI
    "Dallas": "19100",         # Dallas–Fort Worth–Arlington, TX
    "Houston": "26420",        # Houston–The Woodlands–Sugar Land, TX
    "Minneapolis": "33460",    # Minneapolis–St. Paul–Bloomington, MN–WI
    "Phoenix": "38060"         # Phoenix–Mesa–Chandler, AZ
}
INV_CBSA = {v: k for k, v in CBSA_MAP.items()}

CROSSWALK_XLSX = "cbsa_county_rel_2023.xlsx"

In [65]:
# ---------------- HELPERS ----------------
def z2(x): return str(x).zfill(2)
def z3(x): return str(x).zfill(3)

def infer_year_from_fname(fname: str):
    """
    Handle patterns like 1112, 1213, 1516, 2021, 2122...
    We take the later year in the pair or explicit 4-digit year.
    Examples:
      countyinflow1112.csv -> 2012
      countyoutflow2021.csv -> 2021
      countyinflow2122.csv -> 2022
    """
    base = os.path.basename(fname).lower()
    # explicit 4-digit years
    yrs4 = re.findall(r"(20\d{2})", base)
    if yrs4:
        return max(int(y) for y in yrs4)
    # pair 1112, 1213, etc.
    m = re.search(r"(?<!\d)(\d{2})(\d{2})(?!\d)", base)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        a += 2000
        b += 2000
        return max(a, b)
    return None


def load_irs_file(path: str) -> pd.DataFrame:
    """
    Load a single IRS county inflow/outflow file:
    - enforce latin1 encoding
    - lower-case col names
    - build origin_fips / dest_fips
    - convert n1 to numeric
    """
    df = pd.read_csv(path, dtype=str, encoding="latin1")
    df.columns = df.columns.str.lower()

    for col in ["y1_statefips", "y1_countyfips", "y2_statefips", "y2_countyfips"]:
        if col not in df.columns:
            raise ValueError(f"Column {col} missing in {os.path.basename(path)}")

    df["y1_statefips"] = df["y1_statefips"].astype(str)
    df["y1_countyfips"] = df["y1_countyfips"].astype(str)
    df["y2_statefips"] = df["y2_statefips"].astype(str)
    df["y2_countyfips"] = df["y2_countyfips"].astype(str)

    df["origin_fips"] = df["y1_statefips"].map(z2) + df["y1_countyfips"].map(z3)
    df["dest_fips"]   = df["y2_statefips"].map(z2) + df["y2_countyfips"].map(z3)

    df["n1"] = pd.to_numeric(df["n1"], errors="coerce")
    return df


def compute_inflow(df_in: pd.DataFrame, metro_cbsa: str, metro_counties: dict) -> int:
    """
    Households (returns) moving INTO metro: dest in metro, origin outside.
    """
    counties = set(metro_counties[metro_cbsa])
    mask_dest_in_metro  = df_in["dest_fips"].isin(counties)
    mask_origin_outside = ~df_in["origin_fips"].isin(counties)
    return int(df_in.loc[mask_dest_in_metro & mask_origin_outside, "n1"].sum())


def compute_outflow(df_out: pd.DataFrame, metro_cbsa: str, metro_counties: dict) -> int:
    """
    Households (returns) moving OUT OF metro: origin in metro, dest outside.
    """
    counties = set(metro_counties[metro_cbsa])
    mask_origin_in_metro = df_out["origin_fips"].isin(counties)
    mask_dest_outside    = ~df_out["dest_fips"].isin(counties)
    return int(df_out.loc[mask_origin_in_metro & mask_dest_outside, "n1"].sum())

In [51]:
# ---------------- LOAD CROSSWALK & BUILD metro_counties ----------------
cw = pd.read_excel(
    CROSSWALK_XLSX,
    header=2,      # row 3 is header
    dtype=str
)
cw.columns = cw.columns.str.lower().str.strip()

cw = cw.rename(columns={
    "cbsa code": "cbsa_code",
    "cbsa title": "cbsa_title",
    "fips state code": "statefp",
    "fips county code": "countyfp"
})

cw["statefp"] = cw["statefp"].map(z2)
cw["countyfp"] = cw["countyfp"].map(z3)
cw["county_fips"] = cw["statefp"] + cw["countyfp"]

# Only keep counties in our target CBSAs
cw_sub = cw[cw["cbsa_code"].isin(CBSA_MAP.values())].copy()

metro_counties = (
    cw_sub.groupby("cbsa_code")["county_fips"]
    .apply(list)
    .to_dict()
)

In [66]:
# ---------------- FIND INFLOW / OUTFLOW FILES BY YEAR ----------------
all_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]
inflow_files  = [f for f in all_files if "inflow"  in f.lower()]
outflow_files = [f for f in all_files if "outflow" in f.lower()]

inflow_by_year = {}
outflow_by_year = {}

for f in inflow_files:
    year = infer_year_from_fname(f)
    if year is not None:
        inflow_by_year[year] = os.path.join(DATA_DIR, f)

for f in outflow_files:
    year = infer_year_from_fname(f)
    if year is not None:
        outflow_by_year[year] = os.path.join(DATA_DIR, f)

print("Inflow years found:", sorted(inflow_by_year.keys()))
print("Outflow years found:", sorted(outflow_by_year.keys()))

Inflow years found: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Outflow years found: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [67]:
# ---------------- MAIN LOOP: COMPUTE FLOWS FOR EACH METRO & YEAR ----------------
records = []

for year in sorted(inflow_by_year.keys()):
    if year not in outflow_by_year:
        print(f"[WARN] No outflow file found for year {year}, skipping.")
        continue

    print(f"Processing year {year} ...")
    df_in  = load_irs_file(inflow_by_year[year])
    df_out = load_irs_file(outflow_by_year[year])

    for metro_name, cbsa_code in CBSA_MAP.items():
        inflow  = compute_inflow(df_in,  cbsa_code, metro_counties)
        outflow = compute_outflow(df_out, cbsa_code, metro_counties)
        net     = inflow - outflow

        records.append({
            "year": year,
            "metro": metro_name,
            "cbsa_code": cbsa_code,
            "inflow_n1": inflow,
            "outflow_n1": outflow,
            "net_n1": net
        })


Processing year 2012 ...
Processing year 2013 ...
Processing year 2014 ...
Processing year 2015 ...
Processing year 2016 ...
Processing year 2017 ...
Processing year 2018 ...
Processing year 2019 ...
Processing year 2020 ...
Processing year 2021 ...
Processing year 2022 ...


In [68]:
df_metros = pd.DataFrame(records).sort_values(["metro", "year"])

print(df_metros.head())

# ---------------- SAVE TO EXCEL ----------------
out_xlsx = "metro_migration_net_2011_2022.xlsx"
df_metros.to_excel(out_xlsx, index=False)
print(f"Saved: {out_xlsx}")

    year    metro cbsa_code  inflow_n1  outflow_n1  net_n1
0   2012  Atlanta     12060     883953      859266   24687
6   2013  Atlanta     12060     831591      802282   29309
12  2014  Atlanta     12060     774438      736385   38053
18  2015  Atlanta     12060     625011      584046   40965
24  2016  Atlanta     12060     801770      734673   67097
Saved: metro_migration_net_2011_2022.xlsx
